# Sarvam-1 Evaluation on Odia GSM8K

Evaluates `sarvamai/sarvam-1` on the `tripathysagar/odia-gsm8k` dataset.

**Metrics tracked:**
- **Accuracy**: exact-match on the final numerical answer
- **Latency**: TTFT, total generation time, tokens/sec (throughput), time per output token (TPOT), end-to-end latency

## 1. Setup

In [1]:
# Install / upgrade dependencies
# %pip install -q transformers datasets accelerate torch python-dotenv pandas matplotlib seaborn tqdm

In [2]:
import os
import re
import json
import time
import threading
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

# Import comet_ml *before* torch so Comet can auto-instrument the framework.
# (Comet warns and disables some autologging if torch is imported first.)
import comet_ml

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

sns.set_theme(style="whitegrid", palette="muted")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA RTX 4000 Ada Generation


## 2. Configuration

In [3]:
import sys
from pathlib import Path

# Make the repo root importable so `config.py` resolves whether this notebook's
# working dir is the repo root or the notebooks/ subfolder.
_root = Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import *   # all non-secret params (config.py also loads .env secrets)

# config.py stays torch-free (so it can be imported before torch for Comet),
# so resolve the cuda→cpu fallback here.
if DEVICE == "cuda" and not torch.cuda.is_available():
    DEVICE = "cpu"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model        : {MODEL_ID}")
print(f"Dataset      : {DATASET_ID}  (eval={DATASET_SPLIT}, shots={FEW_SHOT_SPLIT})")
print(f"Few-shot     : {NUM_FEW_SHOT}")
print(f"Question col : {QUESTION_COL}")
print(f"Answer col   : {ANSWER_COL}")
print(f"Device       : {DEVICE}")
print(f"Samples      : {NUM_EVAL_SAMPLES if NUM_EVAL_SAMPLES > 0 else 'all'}")

Model        : sarvamai/sarvam-1
Dataset      : tripathysagar/odia-gsm8k  (eval=test, shots=train)
Few-shot     : 3
Question col : odia_question
Answer col   : odia_answer
Device       : cuda
Samples      : 100


## 2b. Experiment Tracking (Comet ML)

Logs each eval run as a Comet experiment in `COMET_EVAL_PROJECT`. Tag with `EVAL_RUN_TAG` (`base` / `sft` / `grpo`) — or leave blank and it's inferred from `MODEL_ID`. Open the project in Comet → "Compare" view to see all three runs side-by-side.

If `COMET_API_KEY` is not set, this is a no-op and the rest of the notebook still works.

In [4]:
# COMET_API_KEY / COMET_WORKSPACE / COMET_EVAL_PROJECT / EVAL_RUN_TAG all come
# from config.py (imported above); EVAL_RUN_TAG is already inferred from MODEL_ID.
experiment = None
if COMET_API_KEY:
    try:
        from comet_ml import Experiment
        experiment = Experiment(
            api_key=COMET_API_KEY,
            workspace=COMET_WORKSPACE or None,
            project_name=COMET_EVAL_PROJECT,
            log_code=False,
            log_graph=False,
            auto_param_logging=False,
            auto_metric_logging=False,
        )
        experiment.set_name(f"{EVAL_RUN_TAG}-{MODEL_ID.split('/')[-1]}")
        # GPU_TYPE tags the run by default (latency metrics are hardware-dependent)
        eval_tags = [EVAL_RUN_TAG, "eval", DATASET_SPLIT]
        if GPU_TYPE:
            eval_tags.append(GPU_TYPE)
        experiment.add_tags(eval_tags)
        experiment.log_parameters({
            "model_id":         MODEL_ID,
            "dataset_id":       DATASET_ID,
            "dataset_split":    DATASET_SPLIT,
            "num_few_shot":     NUM_FEW_SHOT,
            "max_new_tokens":   MAX_NEW_TOKENS,
            "num_eval_samples": NUM_EVAL_SAMPLES,
            "device":           DEVICE,
            "gpu_type":         GPU_TYPE or "(unset)",
            "run_tag":          EVAL_RUN_TAG,
        })
        print(f"Comet experiment started — project={COMET_EVAL_PROJECT}, tags={eval_tags}")
    except Exception as exc:
        print(f"Comet init failed (continuing without tracking): {exc}")
        experiment = None
else:
    print(f"COMET_API_KEY not set — eval will run but won't be logged to Comet. (Run tag would be: {EVAL_RUN_TAG})")

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/paresh-pradhan/sarvam1-odia-gsm8k-eval/149f4734fc0944ff94b3dc5672fdd922



Comet experiment started — project=sarvam1-odia-gsm8k-eval, tags=['base', 'eval', 'test', 'rtx4000ada']


In [5]:
# experiment.end()

## 3. Load Dataset

In [6]:
dataset = load_dataset(DATASET_ID, token=HF_TOKEN)
print(dataset)

eval_data = dataset[DATASET_SPLIT]

if NUM_EVAL_SAMPLES > 0:
    eval_data = eval_data.select(range(min(NUM_EVAL_SAMPLES, len(eval_data))))

print(f"\nUsing split '{DATASET_SPLIT}' — {len(eval_data)} samples")
print("\nSample entry (verify gold-answer format below):")
print(f"Question: {eval_data[0][QUESTION_COL]}")
print(f"Answer  : {eval_data[0][ANSWER_COL]!r}")

DatasetDict({
    test: Dataset({
        features: ['id', 'question', 'answer', 'odia_question', 'odia_answer'],
        num_rows: 1319
    })
    train: Dataset({
        features: ['id', 'question', 'answer', 'odia_question', 'odia_answer'],
        num_rows: 7473
    })
})

Using split 'test' — 100 samples

Sample entry (verify gold-answer format below):
Question: ଜାନେଟଙ୍କ ବତକ ପ୍ରତିଦିନ 16 ଟି ଡିମ୍ବ ଦଦଦଦଦିଅନ୍ତି। ସେ ପ୍ରତି ସକାଳେ ନାସ୍ତା ପାଇଁ 3 ଟି ଖାନ୍ତି ଏବଂ ପ୍ରତିଦିନ 4 ଟି ଦିଆଁ ସାହିମାନଙ୍କ ପାଇଁ ମଫିନ ବେକ କରନ୍ତି। ବାକି ଅଂଶକୁ ସେ ପ୍ରତିଦିନ କୃଷକ ବଜାରରେ ପ୍ରତି ତାଜା ବତକ ଡିମ୍ବରେ $2 ରେ ବିକ୍ରି କରନ୍ତି। କୃଷକ ବଜାରରେ ସେ ପ୍ରତିଦିନ କେତେ ଡଲର ରୋଜଗାର କରନ୍ତି?
Answer  : 'ଜାନେଟ ପ୍ରତିଦିନ 16 - 3 - 4 = <<16-3-4=9>>9 ଟି ବତକ ଡିମ୍ବ ବିକ୍ରି କରନ୍ତି।\nସେ କୃଷକ ବଜାରରେ ପ୍ରତିଦିନ 9 * 2 = $<<9*2=18>>18 ରୋଜଗାର କରନ୍ତି।\n#### 18'


In [7]:
print("Available columns:", eval_data.column_names)
print(f"Question column  : {QUESTION_COL}")
print(f"Answer column    : {ANSWER_COL}")

assert QUESTION_COL in eval_data.column_names, f"QUESTION_COL='{QUESTION_COL}' not found in {eval_data.column_names}"
assert ANSWER_COL   in eval_data.column_names, f"ANSWER_COL='{ANSWER_COL}' not found in {eval_data.column_names}"

Available columns: ['id', 'question', 'answer', 'odia_question', 'odia_answer']
Question column  : odia_question
Answer column    : odia_answer


In [8]:
# Build few-shot examples from a separate split (Sarvam-1 is a base model — few-shot helps a lot)
FEW_SHOT_EXAMPLES = []
if NUM_FEW_SHOT > 0:
    shots = dataset[FEW_SHOT_SPLIT].select(range(NUM_FEW_SHOT))
    FEW_SHOT_EXAMPLES = [
        {"question": s[QUESTION_COL], "answer": str(s[ANSWER_COL])}
        for s in shots
    ]

print(f"Loaded {len(FEW_SHOT_EXAMPLES)} few-shot examples from split '{FEW_SHOT_SPLIT}'")
if FEW_SHOT_EXAMPLES:
    print("\nFirst few-shot example:")
    print(f"Q: {FEW_SHOT_EXAMPLES[0]['question'][:150]}...")
    print(f"A: {FEW_SHOT_EXAMPLES[0]['answer'][:200]}...")

Loaded 3 few-shot examples from split 'train'

First few-shot example:
Q: ନଟାଲିଆ ଏପ୍ରିଲରେ ନିଜର 48 ଜଣ ବନ୍ଧୁଙ୍କୁ କ୍ଲିପ ବିକ୍ରି କଲେ, ଏବଂ ମଇରେ ତାହାଠାରୁ ଅଧା ସଂଖ୍ୟା କ୍ଲିପ ବିକ୍ରି କରିଥିଲେ। ଏପ୍ରିଲ ଓ ମଇ ମିଳାଇ ନଟାଲିଆ ମୋଟ କେତେ କ୍ଲିପ ବିକ୍...
A: ନଟାଲିଆ ମଇରେ 48/2 = <<48/2=24>>24 କ୍ଲିପ ବିକ୍ରି କଲେ।
ନଟାଲିଆ ଏପ୍ରିଲ ଓ ମଇ ମିଳାଇ 48+24 = <<48+24=72>>72 କ୍ଲିପ ବିକ୍ରି କଲେ।
#### 72...


## 4. Load Model & Tokenizer

In [9]:
print(f"Loading tokenizer from {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model from {MODEL_ID} ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16 if DEVICE != "cpu" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else DEVICE,
    trust_remote_code=True,
)
model.eval()
print("Model loaded.")

Loading tokenizer from sarvamai/sarvam-1 ...
Loading model from sarvamai/sarvam-1 ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

Model loaded.


## 5. Helper Functions

In [10]:
SYSTEM_PROMPT = (
    "ଆପଣ ଜଣେ ସହାୟକ ଗଣିତ ସହକାରୀ ଅଟନ୍ତି। "
    "ତଳେ ଦିଆଯାଇଥିବା ସମସ୍ୟାକୁ ପର୍ଯ୍ୟାୟକ୍ରମେ ସମାଧାନ କରନ୍ତୁ। "
    "ଶେଷରେ, ଆପଣଙ୍କର ଚୂଡ଼ାନ୍ତ ସାଂଖ୍ୟିକ ଉତ୍ତରକୁ ଏକ ନୂଆ ଧାଡ଼ିରେ '####' ସହିତ ଆରମ୍ଭ କରି ଲେଖନ୍ତୁ।"
)

# Odia digits (୦-୯) → Arabic digits (0-9). Sarvam-1 may emit numerals in Odia script.
ODIA_TO_ARABIC = str.maketrans("୦୧୨୩୪୫୬୭୮୯", "0123456789")


def normalize_digits(text: str) -> str:
    return text.translate(ODIA_TO_ARABIC)


def build_prompt(question: str) -> str:
    """Format question with optional few-shot context.
    Works for base (few-shot) and SFT/GRPO (set NUM_FEW_SHOT=0) models —
    the SFT/GRPO format matches the zero-shot fallback exactly."""
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for ex in FEW_SHOT_EXAMPLES:
            messages.append({"role": "user",      "content": ex["question"]})
            messages.append({"role": "assistant", "content": ex["answer"]})
        messages.append({"role": "user", "content": question})
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Base-model / SFT / GRPO fallback: plain text format
    parts = [SYSTEM_PROMPT, ""]
    for ex in FEW_SHOT_EXAMPLES:
        parts.append(f"ପ୍ରଶ୍ନ: {ex['question']}")
        parts.append(f"ଉତ୍ତର: {ex['answer']}")
        parts.append("")
    parts.append(f"ପ୍ରଶ୍ନ: {question}")
    parts.append("ଉତ୍ତର:")
    return "\n".join(parts)


def extract_numerical_answer(text: str) -> Optional[float]:
    """Extract the final numerical answer; handles both Arabic and Odia digits."""
    text = normalize_digits(text)
    # GSM8K convention: #### <number>
    marker_match = re.search(r"####\s*([\-\d,\.]+)", text)
    if marker_match:
        num_str = marker_match.group(1).replace(",", "").rstrip(".")
        try:
            return float(num_str)
        except ValueError:
            pass
    # Fallback: last standalone number in text
    numbers = re.findall(r"-?\d+(?:[,\.]\d+)*", text)
    if numbers:
        try:
            return float(numbers[-1].replace(",", "").rstrip("."))
        except ValueError:
            pass
    return None


def answers_match(pred: Optional[float], gold: Optional[float], tol: float = 1e-3) -> bool:
    if pred is None or gold is None:
        return False
    return abs(pred - gold) <= tol * max(1.0, abs(gold))


# --- Correctness side-signals: format + language adherence ---

_FORMAT_RE = re.compile(r"####\s*-?\d")


def has_format_marker(text: str) -> bool:
    """True iff output contains '#### <number>' (Arabic or Odia digits) — the GSM8K answer convention."""
    return bool(_FORMAT_RE.search(normalize_digits(text)))


def odia_script_ratio(text: str) -> float:
    """Fraction of *script* characters in Odia (vs Latin). Ignores digits, punctuation, whitespace.
    Higher = better Odia adherence; low values mean the model code-switched to English."""
    odia  = sum(1 for c in text if "\u0B00" <= c <= "\u0B7F" and c.isalpha())
    latin = sum(1 for c in text if c.isascii() and c.isalpha())
    total = odia + latin
    return odia / total if total > 0 else 0.0

In [11]:
@dataclass
class LatencyRecord:
    prompt_tokens: int = 0
    output_tokens: int = 0
    ttft_ms: float = 0.0          # time to first token (chunk)
    total_latency_ms: float = 0.0 # wall-clock end-to-end
    throughput_tok_per_sec: float = 0.0
    tpot_ms: float = 0.0          # avg time per generated token (excluding first)


def _cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def generate_with_metrics(prompt: str) -> tuple[str, LatencyRecord]:
    """Greedy decode + latency stats. Token count comes from generate(), not text re-encoding."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_tokens = inputs["input_ids"].shape[-1]

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=True,
        streamer=streamer,
    )

    gen_out: dict = {}

    def _run_generation():
        with torch.no_grad():
            gen_out["result"] = model.generate(**gen_kwargs)

    _cuda_sync()
    t_start = time.perf_counter()

    thread = threading.Thread(target=_run_generation)
    thread.start()

    generated_chunks: list[str] = []
    ttft_ms: Optional[float] = None
    for token_text in streamer:
        if ttft_ms is None:
            ttft_ms = (time.perf_counter() - t_start) * 1000
        generated_chunks.append(token_text)

    thread.join()
    _cuda_sync()
    total_latency_ms = (time.perf_counter() - t_start) * 1000

    full_output = "".join(generated_chunks)

    # Accurate output-token count from generate(), not text re-encoding
    result = gen_out.get("result")
    if result is not None:
        output_tokens = int(result.sequences.shape[-1] - prompt_tokens)
    else:
        output_tokens = len(tokenizer.encode(full_output, add_special_tokens=False))

    if ttft_ms is None:
        ttft_ms = total_latency_ms

    # TPOT = inter-token latency = (total - ttft) / (output_tokens - 1)
    if output_tokens > 1:
        tpot_ms = (total_latency_ms - ttft_ms) / (output_tokens - 1)
    else:
        tpot_ms = total_latency_ms

    throughput = output_tokens / (total_latency_ms / 1000) if total_latency_ms > 0 else 0.0

    return full_output, LatencyRecord(
        prompt_tokens=prompt_tokens,
        output_tokens=output_tokens,
        ttft_ms=ttft_ms,
        total_latency_ms=total_latency_ms,
        throughput_tok_per_sec=throughput,
        tpot_ms=tpot_ms,
    )

In [12]:
results = []

try:
    for idx, sample in enumerate(tqdm(eval_data, desc="Evaluating")):
        question = sample[QUESTION_COL]
        gold_raw = sample[ANSWER_COL]
        gold_num = extract_numerical_answer(str(gold_raw))

        prompt = build_prompt(question)

        try:
            output_text, latency = generate_with_metrics(prompt)
        except Exception as exc:
            print(f"[{idx}] Generation error: {exc}")
            output_text = ""
            latency = LatencyRecord()

        pred_num   = extract_numerical_answer(output_text)
        correct    = answers_match(pred_num, gold_num)
        format_ok  = has_format_marker(output_text)
        odia_ratio = odia_script_ratio(output_text)

        results.append({
            "idx":               idx,
            "question":          question,
            "gold_raw":          str(gold_raw),
            "gold_num":          gold_num,
            "output_text":       output_text,
            "pred_num":          pred_num,
            "correct":           correct,
            "format_ok":         format_ok,
            "odia_ratio":        odia_ratio,
            "prompt_tokens":     latency.prompt_tokens,
            "output_tokens":     latency.output_tokens,
            "ttft_ms":           latency.ttft_ms,
            "total_latency_ms":  latency.total_latency_ms,
            "tpot_ms":           latency.tpot_ms,
            "throughput_tok_s":  latency.throughput_tok_per_sec,
        })
except BaseException as exc:
    # Mark the Comet run as failed so it doesn't sit in "running" state forever.
    # BaseException catches KeyboardInterrupt too — partial runs from Ctrl-C also get cleaned up.
    if experiment is not None:
        try:
            experiment.log_other("status", "failed")
            experiment.log_other("failure_reason", f"{type(exc).__name__}: {str(exc)[:500]}")
            experiment.log_other("samples_completed", len(results))
            experiment.end()
        except Exception:
            pass
    raise

df = pd.DataFrame(results)
print(f"\nEvaluation complete — {len(df)} samples")

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug


Evaluation complete — 100 samples


In [13]:
# Warmup: amortize CUDA init / kernel compilation so the first eval sample isn't an outlier
print("Warming up generation...")
_warm_prompt = build_prompt(eval_data[0][QUESTION_COL])
_, _warm_lat = generate_with_metrics(_warm_prompt)
print(f"Warmup done — {_warm_lat.total_latency_ms:.0f} ms, {_warm_lat.output_tokens} tokens")

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Warming up generation...
Warmup done — 3944 ms, 175 tokens


In [14]:
total   = len(df)
correct = int(df["correct"].sum())
no_pred = int(df["pred_num"].isna().sum())
accuracy = correct / total * 100

# Side-signals
format_ok_pct       = df["format_ok"].mean() * 100
odia_ratio_mean     = df["odia_ratio"].mean() * 100
odia_pure_pct       = (df["odia_ratio"] >= 0.95).mean() * 100   # ≥95% Odia script
english_drift_pct   = (df["odia_ratio"] <  0.50).mean() * 100   # majority Latin = drift

# Generation-cap utilization — samples truncated at MAX_NEW_TOKENS lose their
# '#### N' line, which depresses format adherence + accuracy and clamps the
# output_tokens / latency tails. High value → raise MAX_NEW_TOKENS in config.py.
cap_hit_pct = (df["output_tokens"] >= MAX_NEW_TOKENS).mean() * 100

# Decompose accuracy: how much correctness is hidden behind format/language failures?
acc_among_formatted = df.loc[df["format_ok"], "correct"].mean() * 100 if df["format_ok"].any() else 0.0

print("=" * 48)
print(f"Total samples         : {total}")
print(f"Accuracy (final ans)  : {accuracy:.2f}%   ({correct}/{total})")
print(f"  └─ among format-ok  : {acc_among_formatted:.2f}%")
print(f"No answer found       : {no_pred} ({no_pred/total*100:.1f}%)")
print("-" * 48)
print(f"Format adherence      : {format_ok_pct:.2f}%   (output contains '#### N')")
print(f"Hit token cap ({MAX_NEW_TOKENS}) : {cap_hit_pct:.1f}%  (truncated → may lose '#### N')")
print(f"Odia script ratio     : {odia_ratio_mean:.2f}%  (mean over outputs)")
print(f"  ≥95% Odia (pure)    : {odia_pure_pct:.2f}%")
print(f"  <50% Odia (drifted) : {english_drift_pct:.2f}%")
print("=" * 48)

Total samples         : 100
Accuracy (final ans)  : 12.00%   (12/100)
  └─ among format-ok  : 31.82%
No answer found       : 0 (0.0%)
------------------------------------------------
Format adherence      : 22.00%   (output contains '#### N')
Hit token cap (1024) : 10.0%  (truncated → may lose '#### N')
Odia script ratio     : 96.02%  (mean over outputs)
  ≥95% Odia (pure)    : 90.00%
  <50% Odia (drifted) : 2.00%


## 7. Accuracy Metrics

In [15]:
total   = len(df)
correct = df["correct"].sum()
no_pred = df["pred_num"].isna().sum()
accuracy = correct / total * 100

print("=" * 40)
print(f"Total samples   : {total}")
print(f"Correct         : {correct}")
print(f"Accuracy        : {accuracy:.2f}%")
print(f"No answer found : {no_pred} ({no_pred/total*100:.1f}%)")
print("=" * 40)

Total samples   : 100
Correct         : 12
Accuracy        : 12.00%
No answer found : 0 (0.0%)


## 8. Latency Metrics

In [16]:
latency_cols = ["ttft_ms", "total_latency_ms", "tpot_ms", "throughput_tok_s", "output_tokens"]
lat_summary  = df[latency_cols].describe(percentiles=[0.5, 0.75, 0.95, 0.99])

print("\n--- Latency Summary ---")
print(lat_summary.T[["mean", "50%", "75%", "95%", "99%", "min", "max"]].to_string())


--- Latency Summary ---
                         mean          50%          75%           95%           99%          min           max
ttft_ms             68.486074    63.977648    64.794511     67.233757     74.356256    57.028122    592.792332
total_latency_ms  7011.211304  4988.198120  7709.216783  22820.174075  22989.378171  1357.654035  22993.895493
tpot_ms             22.384293    22.313681    22.438023     22.866598     23.301293    22.123046     23.311025
throughput_tok_s    44.198185    44.373001    44.589518     44.887470     44.933312    38.856221     45.026453
output_tokens      311.330000   221.500000   342.500000   1024.000000   1024.000000    59.000000   1024.000000


## 9. Visualizations

In [17]:
csv_path = RESULTS_DIR / "eval_results.csv"
df.to_csv(csv_path, index=False)
print(f"Raw results saved to {csv_path}")

summary = {
    "model":                MODEL_ID,
    "dataset":              DATASET_ID,
    "split":                DATASET_SPLIT,
    "num_few_shot":         NUM_FEW_SHOT,
    "num_samples":          total,
    "accuracy_pct":         round(accuracy, 4),
    "accuracy_among_formatted_pct": round(acc_among_formatted, 4),
    "no_answer_pct":        round(no_pred / total * 100, 4),
    "format_adherence_pct": round(format_ok_pct, 4),
    "odia_ratio_mean_pct":  round(odia_ratio_mean, 4),
    "odia_pure_pct":        round(odia_pure_pct, 4),
    "english_drift_pct":    round(english_drift_pct, 4),
    "latency": {
        "ttft_ms":          {p: round(np.percentile(df["ttft_ms"], p), 2) for p in [50, 95, 99]},
        "e2e_ms":           {p: round(np.percentile(df["total_latency_ms"], p), 2) for p in [50, 95, 99]},
        "tpot_ms":          {p: round(np.percentile(df["tpot_ms"], p), 2) for p in [50, 95, 99]},
        "throughput_tok_s": {"mean": round(df["throughput_tok_s"].mean(), 2),
                             "p50":  round(df["throughput_tok_s"].median(), 2)},
    },
}

summary_path = RESULTS_DIR / "eval_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"Summary saved to {summary_path}")
print(json.dumps(summary, indent=2))

# --- Push to Comet for cross-run comparison ---
if experiment is not None:
    # Eval is a single snapshot (no training loop, auto-logging off), so Comet's
    # curr_step is None. log_histogram_3d / log_asset_data require a step, so pin
    # one explicitly before logging.
    experiment.set_step(0)

    # Flat metrics → Comet's compare view ranks/plots these across runs
    flat_metrics = {
        "accuracy_pct":                  summary["accuracy_pct"],
        "accuracy_among_formatted_pct":  summary["accuracy_among_formatted_pct"],
        "no_answer_pct":                 summary["no_answer_pct"],
        "format_adherence_pct":          summary["format_adherence_pct"],
        "odia_ratio_mean_pct":           summary["odia_ratio_mean_pct"],
        "odia_pure_pct":                 summary["odia_pure_pct"],
        "english_drift_pct":             summary["english_drift_pct"],
        "num_samples":                   summary["num_samples"],
        "ttft_ms_p50":                   summary["latency"]["ttft_ms"][50],
        "ttft_ms_p95":                   summary["latency"]["ttft_ms"][95],
        "ttft_ms_p99":                   summary["latency"]["ttft_ms"][99],
        "e2e_ms_p50":                    summary["latency"]["e2e_ms"][50],
        "e2e_ms_p95":                    summary["latency"]["e2e_ms"][95],
        "e2e_ms_p99":                    summary["latency"]["e2e_ms"][99],
        "tpot_ms_p50":                   summary["latency"]["tpot_ms"][50],
        "tpot_ms_p95":                   summary["latency"]["tpot_ms"][95],
        "throughput_tok_s_mean":         summary["latency"]["throughput_tok_s"]["mean"],
        "throughput_tok_s_p50":          summary["latency"]["throughput_tok_s"]["p50"],
    }
    experiment.log_metrics(flat_metrics)

    # Per-sample distributions (Comet renders histograms)
    for col in ["ttft_ms", "total_latency_ms", "tpot_ms", "throughput_tok_s", "output_tokens", "odia_ratio"]:
        experiment.log_histogram_3d(df[col].dropna().tolist(), name=col, step=0)

    # Assets: full results CSV, summary JSON, plot
    experiment.log_asset(str(csv_path),     file_name="eval_results.csv")
    experiment.log_asset(str(summary_path), file_name="eval_summary.json")
    plot_path = RESULTS_DIR / "eval_plots.png"
    if plot_path.exists():
        experiment.log_image(str(plot_path), name="eval_plots")

    # Per-sample table (Comet "Data" tab) — drop the long output_text to keep the table light
    table_df = df.drop(columns=["output_text"], errors="ignore").copy()
    experiment.log_table("per_sample_results.csv", tabular_data=table_df)

    experiment.end()
    print(f"\nLogged eval run to Comet — tag='{EVAL_RUN_TAG}', project='{COMET_EVAL_PROJECT}'")
    print("Open the project in Comet and use 'Compare' to overlay base/sft/grpo runs.")

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : base-sarvam-1
COMET INFO:     url                   : https://www.comet.com/paresh-pradhan/sarvam1-odia-gsm8k-eval/149f4734fc0944ff94b3dc5672fdd922
COMET INFO:   Metrics:
COMET INFO:     accuracy_among_formatted_pct : 31.8182
COMET INFO:     accuracy_pct                 : 12.0
COMET INFO:     e2e_ms_p50                   : 4988.2
COMET INFO:     e2e_ms_p95                   : 22820.17
COMET INFO:     e2e_ms_p99                   : 22989.38
COMET INFO:     english_drift_pct            : 2.0
COMET INFO:     format_adherence_pct         : 22.0
COMET INFO:     no_answer_pct                : 0.0
COMET INFO:     num_samples                  : 100
COMET INF

Raw results saved to results/eval_results.csv
Summary saved to results/eval_summary.json
{
  "model": "sarvamai/sarvam-1",
  "dataset": "tripathysagar/odia-gsm8k",
  "split": "test",
  "num_few_shot": 3,
  "num_samples": 100,
  "accuracy_pct": 12.0,
  "accuracy_among_formatted_pct": 31.8182,
  "no_answer_pct": 0.0,
  "format_adherence_pct": 22.0,
  "odia_ratio_mean_pct": 96.0224,
  "odia_pure_pct": 90.0,
  "english_drift_pct": 2.0,
  "latency": {
    "ttft_ms": {
      "50": 63.98,
      "95": 67.23,
      "99": 74.36
    },
    "e2e_ms": {
      "50": 4988.2,
      "95": 22820.17,
      "99": 22989.38
    },
    "tpot_ms": {
      "50": 22.31,
      "95": 22.87,
      "99": 23.3
    },
    "throughput_tok_s": {
      "mean": 44.2,
      "p50": 44.37
    }
  }
}


COMET INFO: Please wait for assets to finish uploading (timeout is 10800 seconds)
COMET INFO: All assets have been sent, waiting for delivery confirmation



Logged eval run to Comet — tag='base', project='sarvam1-odia-gsm8k-eval'
Open the project in Comet and use 'Compare' to overlay base/sft/grpo runs.


## 10. Save Results

In [18]:
csv_path = RESULTS_DIR / "eval_results.csv"
df.to_csv(csv_path, index=False)
print(f"Raw results saved to {csv_path}")

summary = {
    "model":                MODEL_ID,
    "dataset":              DATASET_ID,
    "split":                DATASET_SPLIT,
    "num_few_shot":         NUM_FEW_SHOT,
    "num_samples":          total,
    "accuracy_pct":         round(accuracy, 4),
    "no_answer_pct":        round(no_pred / total * 100, 4),
    "latency": {
        "ttft_ms":          {p: round(np.percentile(df["ttft_ms"], p), 2) for p in [50, 95, 99]},
        "e2e_ms":           {p: round(np.percentile(df["total_latency_ms"], p), 2) for p in [50, 95, 99]},
        "tpot_ms":          {p: round(np.percentile(df["tpot_ms"], p), 2) for p in [50, 95, 99]},
        "throughput_tok_s": {"mean": round(df["throughput_tok_s"].mean(), 2),
                             "p50":  round(df["throughput_tok_s"].median(), 2)},
    },
}

summary_path = RESULTS_DIR / "eval_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"Summary saved to {summary_path}")
print(json.dumps(summary, indent=2))

Raw results saved to results/eval_results.csv
Summary saved to results/eval_summary.json
{
  "model": "sarvamai/sarvam-1",
  "dataset": "tripathysagar/odia-gsm8k",
  "split": "test",
  "num_few_shot": 3,
  "num_samples": 100,
  "accuracy_pct": 12.0,
  "no_answer_pct": 0.0,
  "latency": {
    "ttft_ms": {
      "50": 63.98,
      "95": 67.23,
      "99": 74.36
    },
    "e2e_ms": {
      "50": 4988.2,
      "95": 22820.17,
      "99": 22989.38
    },
    "tpot_ms": {
      "50": 22.31,
      "95": 22.87,
      "99": 23.3
    },
    "throughput_tok_s": {
      "mean": 44.2,
      "p50": 44.37
    }
  }
}


## 11. Error Analysis

In [19]:
# Inspect a few incorrect predictions
errors = df[~df["correct"]].head(5)
for _, row in errors.iterrows():
    print(f"--- Sample {row['idx']} ---")
    print(f"Question : {row['question'][:120]}...")
    print(f"Gold     : {row['gold_num']}  (raw: {row['gold_raw'][:60]})")
    print(f"Predicted: {row['pred_num']}")
    print(f"Output   : {row['output_text'][:200]}...")
    print()

--- Sample 0 ---
Question : ଜାନେଟଙ୍କ ବତକ ପ୍ରତିଦିନ 16 ଟି ଡିମ୍ବ ଦଦଦଦଦିଅନ୍ତି। ସେ ପ୍ରତି ସକାଳେ ନାସ୍ତା ପାଇଁ 3 ଟି ଖାନ୍ତି ଏବଂ ପ୍ରତିଦିନ 4 ଟି ଦିଆଁ ସାହିମାନଙ୍କ ...
Gold     : 18.0  (raw: ଜାନେଟ ପ୍ରତିଦିନ 16 - 3 - 4 = <<16-3-4=9>>9 ଟି ବତକ ଡିମ୍ବ ବିକ୍ର)
Predicted: 88.0
Output   : ଜେନେଟ୍ ଙ୍କ ପାଖରେ ୧୬ଟି ଡେଙ୍ଗୁ ଅଛନ୍ତି, ତେଣୁ ସେ ଦିନକୁ ୧୬ x ୩ = ୪୮ଟି ଡେଙ୍ଗୁ ଖାଦ୍ୟ ଖାଇଥାନ୍ତି।
ସେ ପ୍ରତିଦିନ ୪ଟି ଡେଙ୍ଗୁ ମଧ୍ୟ ଦିଅନ୍ତି, ତେଣୁ ସେ ପ୍ରତିଦିନ ୪୮ - ୪ = ୪୪ଟି ଡେଙ୍ଗୁ ଦିଅନ୍ତି।
ପ୍ରତ୍ୟେକ ଡେଙ୍ଗୁ ପାଇଁ, ସେ ୨ଟ...

--- Sample 1 ---
Question : ଏକ ବସ୍ତ୍ର ନିର୍ମାଣରେ 2 ବୋଲ୍ଟ ନୀଳ ଫାଇବର ଏବଂ ତାହାର ଅଧା ପରିମାଣ ଧଳା ଫାଇବର ଲାଗେ। ମୋଟ କେତେ ବୋଲ୍ଟ ଲାଗେ?...
Gold     : 3.0  (raw: ଏହାକୁ 2/2=<<2/2=1>>1 ବୋଲ୍ଟ ଧଳା ଫାଇବର ଲାଗେ।
ତେଣୁ ମୋଟ କପଡ଼ା 2+)
Predicted: 4.0
Output   : ଆସନ୍ତୁ ଅନୁମାନ କରିବା ଯେ ପ୍ରତ୍ୟେକ ବୋଲ୍ଟରେ ସମାନ ପରିମାଣର ଫାଇବର ରହିଛି। ତେଣୁ, ୨ ବୋଲ୍ଟ ନୀଳ ଫାଇବର ଏବଂ ୧ ବୋଲ୍ଟ ଧଳା ଫାଇବର ଆବଶ୍ୟକ ହେବ।

ନୀଳ ରଙ୍ଗର ଫାଇବର ପାଇଁ: ୨ ବୋଲ୍ଟ * ୨ ବୋଲ୍ଟ = ୪ ବୋଲ୍ଟ
ଧଳା ରଙ୍ଗର ଫାଇବର ପାଇଁ: ୧ ବ...

--- Sample 2 ---
Question : ଜଶ ଏକ ଘର ଫ୍ଲିପ କରିବାକୁ ଚେଷ୍ଟା କରନ୍ତି। ସେ $80,000 ରେ ଘର କିଣି $